In [8]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_parquet('data/cleaned_reviews.parquet')

In [5]:
df.head(10)

,Id,Title,Price,User_id,profileName,score,time,summary,text,description,...,image,previewLink,publisher,publishedDate,infoLink,categories,ratingsCount,clean_text,clean_summary,clean_authors
0,0791054608,Guide to Owning a Birman Cat (Popular Cat Libr...,Unknown,A2E3F04ZK7FG66,calvinnme,5.0,1167609600,Great advice on caring for your Birman,I bought this to help understand care issues s...,"A guide to the history, feeding, grooming, exh...",...,http://books.google.com/books/content?id=yyBPG...,http://books.google.nl/books?id=yyBPGwAACAAJ&d...,Chelsea House Pub,1999-01,http://books.google.nl/books?id=yyBPGwAACAAJ&d...,[Juvenile Nonfiction],3,bought help understand care issues specific bi...,great advice caring birman,[karen cummings]
1,0791054608,Guide to Owning a Birman Cat (Popular Cat Libr...,Unknown,A3778VKGYVZ76Q,Valerie Caliendo Kotas (Katacali Cattery),5.0,960163200,The Guide to Owning a Birman Cat (The Guide to...,Wonderful! Karen Cummings writes a book that t...,"A guide to the history, feeding, grooming, exh...",...,http://books.google.com/books/content?id=yyBPG...,http://books.google.nl/books?id=yyBPGwAACAAJ&d...,Chelsea House Pub,1999-01,http://books.google.nl/books?id=yyBPGwAACAAJ&d...,[Juvenile Nonfiction],3,wonderful karen cummings writes book tells bas...,guide owning birman cat guide owning,[karen cummings]
2,0791054608,Guide to Owning a Birman Cat (Popular Cat Libr...,Unknown,Unknown,Unknown,5.0,960163200,"Karen Cummings, comes through.",Wonderful! Karen Cummings writes a book that t...,"A guide to the history, feeding, grooming, exh...",...,http://books.google.com/books/content?id=yyBPG...,http://books.google.nl/books?id=yyBPGwAACAAJ&d...,Chelsea House Pub,1999-01,http://books.google.nl/books?id=yyBPGwAACAAJ&d...,[Juvenile Nonfiction],3,wonderful karen cummings writes book tells the...,karen cummings comes,[karen cummings]
3,006000486X,Tess and the Highlander,Unknown,A2VCGJLKGK2WJJ,Rebecca Herman,5.0,1035244800,My new favorite book from the Avon True Romanc...,Tess was washed ashore on the Isle of May duri...,"In 1543, on a windswept isle off of Scotland, ...",...,http://books.google.com/books/content?id=VmCRS...,http://books.google.nl/books?id=VmCRSPmY3WkC&d...,Harper Collins,2002-11,http://books.google.nl/books?id=VmCRSPmY3WkC&d...,[Juvenile Fiction],17,tess washed ashore isle may terrible storm ele...,new favorite book avon true romance series,[may mcgoldrick]
4,006000486X,Tess and the Highlander,Unknown,Unknown,Unknown,5.0,1041638400,An awesome novel from May McGoldrick,"After enjoying a previous Avon True Romance, G...","In 1543, on a windswept isle off of Scotland, ...",...,http://books.google.com/books/content?id=VmCRS...,http://books.google.nl/books?id=VmCRSPmY3WkC&d...,Harper Collins,2002-11,http://books.google.nl/books?id=VmCRSPmY3WkC&d...,[Juvenile Fiction],17,enjoying previous avon true romance gwyneth th...,awesome novel may mcgoldrick,[may mcgoldrick]
5,006000486X,Tess and the Highlander,Unknown,AVWFMN5CELC8Q,sarah,4.0,1082592000,a great book for Historical romance lovers,This is an engaging a count of life of Tess a ...,"In 1543, on a windswept isle off of Scotland, ...",...,http://books.google.com/books/content?id=VmCRS...,http://books.google.nl/books?id=VmCRSPmY3WkC&d...,Harper Collins,2002-11,http://books.google.nl/books?id=VmCRSPmY3WkC&d...,[Juvenile Fiction],17,engaging count life tess girl young age washed...,great book historical romance lovers,[may mcgoldrick]
6,006000486X,Tess and the Highlander,Unknown,A37XYM3KSEIDLS,"jaina_solo ""jaina_solo""",5.0,1043625600,Loved it!,This book was a perfect historical romance for...,"In 1543, on a windswept isle off of Scotland, ...",...,http://books.google.com/books/content?id=VmCRS...,http://books.google.nl/books?id=VmCRSPmY3WkC&d...,Harper Collins,2002-11,http://books.google.nl/books?id=VmCRSPmY3WkC&d...,[Juvenile Fiction],17,book perfect historical romance teens set scot...,loved,[may mcgoldrick]
7,006000486X,Tess and the Highlander,Unknown,A1IQK80

In [11]:
def eh_desconhecido(x):
    if x is None: return True
    if isinstance(x, (list, np.ndarray)):
        return len(x) == 0 or 'unknown' in x
    return str(x).lower() in ["unknown", "['unknown']", "[]"]

# 3. Cria o gabarito pegando apenas títulos que possuem autores REAIS
gabarito = (
    df[~df["authors"].apply(eh_desconhecido)][["Title", "authors"]]
    .drop_duplicates(subset=["Title"])
    .rename(columns={"authors": "authors_gabarito"})
)

# 4. Faz o cruzamento para testar a recuperação
df_recuperado = df.merge(gabarito, on="Title", how="left")

# 5. Filtra quem era desconhecido mas achou um par no gabarito
linhas_salvas = df_recuperado[
    df_recuperado["authors"].apply(eh_desconhecido) & 
    df_recuperado["authors_gabarito"].notna()
]

print(f"Total de registros que conseguimos recuperar: {len(linhas_salvas)}")
if len(linhas_salvas) > 0:
    print("\n--- Exemplos de Títulos Recuperados (Antes vs Depois) ---")
    print(linhas_salvas[["Title", "authors", "authors_gabarito"]].drop_duplicates(subset=["Title"]).head(10))
else:
    print("\nPoxa, nenhum título cruzou. Significa que os livros com 'unknown' são exclusivos e não possuem duplicatas com autor preenchido.")

Total de registros que conseguimos recuperar: 0

Poxa, nenhum título cruzou. Significa que os livros com 'unknown' são exclusivos e não possuem duplicatas com autor preenchido.


In [6]:
df['clean_authors'].explode().value_counts()

clean_authors
unknown                    390842
jane austen                 37511
j r r tolkien               37302
charles dickens             21878
emily bronte                18445
                            ...  
warren w vache                  1
ann irwin                       1
javier perez de cuellar         1
joseph beam                     1
ian olver                       1
Name: count, Length: 151297, dtype: int64

In [9]:
# 1. Filtra as linhas onde 'unknown' está presente na lista
df_filtrado = df[df['clean_authors'].apply(lambda x: 'unknown' in x if isinstance(x, (list, np.ndarray)) else False)][['authors', 'clean_authors']]

# 2. Converte para tupla apenas o que for iterável, preservando nulos, e roda o distinct
df_filtrado.map(lambda x: tuple(x) if isinstance(x, (list, np.ndarray)) else x).drop_duplicates()

,authors,clean_authors
42,None,"(unknown,)"


In [ ]:
df['categories'].explode().value_counts()

categories
Fiction                         824439
Juvenile Fiction                207542
Biography & Autobiography       107791
Religion                         98035
History                          89988
                                 ...  
Suffolk County (N.Y.)                1
Human body (Philosophy)              1
Art in mathematics education         1
Arkansas River                       1
B-1 bomber                           1
Name: count, Length: 10883, dtype: int64

In [2]:
df_categorie_normalize = pd.read_parquet('data/cleaned_reviews_mapped.parquet')

In [3]:
df_categorie_normalize['categories_clean'].value_counts()

categories_clean
fiction                       845901
juvenile fiction              213825
history                       126627
religion                      112783
biography  autobiography      110010
business  economics            75807
computers                      49194
family  relationships          40905
cooking                        34815
unknown                        33258
education                      32669
social science                 32636
young adult fiction            31858
self-help                      29636
juvenile nonfiction            29484
body mind  spirit              29359
health  fitness                28517
psychology                     28000
science                        27867
travel                         25618
philosophy                     25336
sports  recreation             25006
political science              24318
art                            24262
pets                           22645
adventure stories              20803
drama                

In [4]:
df_pares = df_categorie_normalize[['categories', 'categories_clean']].drop_duplicates()
amostra_notebook = df_pares.sample(n=20, random_state=42)

# Exibe a amostra para você avaliar visualmente
for i, row in enumerate(amostra_notebook.itertuples(), 1):
    print(f"{i}. Original: '{row.categories}'  ==>  Predito: '{row.categories_clean}'")

1. Original: 'electronic government information'  ==>  Predito: 'computers'
2. Original: 'invasion from mars radio play'  ==>  Predito: 'music'
3. Original: 'soccer'  ==>  Predito: 'sports  recreation'
4. Original: 'funeral homes'  ==>  Predito: 'unknown'
5. Original: 'macram'  ==>  Predito: 'unknown'
6. Original: 'interfaith marriage'  ==>  Predito: 'family  relationships'
7. Original: 'south dakota'  ==>  Predito: 'education'
8. Original: 'atlanta ga'  ==>  Predito: 'unknown'
9. Original: 'shock'  ==>  Predito: 'humor'
10. Original: 'doll clothes'  ==>  Predito: 'crafts  hobbies'
11. Original: 'eighteenth century'  ==>  Predito: 'juvenile fiction'
12. Original: 'contests'  ==>  Predito: 'games  activities'
13. Original: 'australians'  ==>  Predito: 'pets'
14. Original: 'amazon river region'  ==>  Predito: 'unknown'
15. Original: 'psychiatric hospital care'  ==>  Predito: 'medical'
16. Original: 'book of mormon'  ==>  Predito: 'bibles'
17. Original: 'local author'  ==>  Predito: 'amer

1. Original: 'electronic government information'  ==>  Predito: 'computers' ✅
2. Original: 'invasion from mars radio play'  ==>  Predito: 'music' ❌
3. Original: 'soccer'  ==>  Predito: 'sports  recreation' ✅
4. Original: 'funeral homes'  ==>  Predito: 'unknown'❌
5. Original: 'macram'  ==>  Predito: 'unknown'❌
6. Original: 'interfaith marriage'  ==>  Predito: 'family  relationships' ✅
7. Original: 'south dakota'  ==>  Predito: 'education'❌
8. Original: 'atlanta ga'  ==>  Predito: 'unknown'❌
9. Original: 'shock'  ==>  Predito: 'humor'❌
10. Original: 'doll clothes'  ==>  Predito: 'crafts  hobbies' ✅
11. Original: 'eighteenth century'  ==>  Predito: 'juvenile fiction'❌
12. Original: 'contests'  ==>  Predito: 'games  activities'❌
13. Original: 'australians'  ==>  Predito: 'pets'❌
14. Original: 'amazon river region'  ==>  Predito: 'unknown'❌
15. Original: 'psychiatric hospital care'  ==>  Predito: 'medical' ✅
16. Original: 'book of mormon'  ==>  Predito: 'bibles' ✅
17. Original: 'local author'  ==>  Predito: 'american literature'❌
18. Original: 'saltcellars'  ==>  Predito: 'unknown'❌
19. Original: 'investments'  ==>  Predito: 'business  economics' ✅
20. Original: 'buddhist logic'  ==>  Predito: 'philosophy' ✅


Assertividade duvidosa, necessario melhor analise de threshold ou avaliação de mtodo para similaridade(talvez outra LLM)

In [ ]:
df_categorie_normalize = pd.read_parquet('data/cleaned_reviews_mapped.parquet')